# GPT-2 and GPT-3 Basic Architecture in NumPy

**Goal for the demo:** build the core pieces of a GPT-style decoder-only Transformer using only NumPy.

This is an educational notebook for Section 1 of the journal club. It is intentionally small and readable. It does **not** train a real GPT-2 or GPT-3 model. Instead, it shows the architecture that GPT-2 and GPT-3 share:

- token embeddings and positional embeddings
- causal self-attention
- multi-head attention
- feed-forward / MLP layers
- residual connections
- layer normalization
- next-token logits
- autoregressive generation

**Teaching note:** the outputs from this notebook are random because the model weights are random. The architecture is real; the intelligence comes from large-scale training.


## 1. Imports and Reproducibility

Only NumPy is required. The fixed seed makes the live demo repeatable.


In [ ]:
from dataclasses import dataclass
import math
import numpy as np

np.set_printoptions(precision=3, suppress=True)
np.random.seed(7)


## 2. Big Picture: GPT-2 and GPT-3 Use the Same Basic Design

GPT-2 and GPT-3 are both **decoder-only Transformers** trained with a simple objective: predict the next token from previous tokens.

The main difference is scale, not a completely different architecture.

| Model family | Architecture | Example published scale | What changes in this notebook |
|---|---|---:|---|
| GPT-2 small | decoder-only Transformer | 12 layers, 768 hidden size, 12 heads | tiny GPT-2-like config |
| GPT-3 175B | decoder-only Transformer | 96 layers, 12,288 hidden size, 96 heads | tiny GPT-3-like config, deeper and wider |

We keep the toy models tiny so every step is visible and fast.


In [ ]:
@dataclass
class GPTConfig:
    """Configuration for a decoder-only GPT-style Transformer."""

    vocab_size: int  # Number of possible token IDs.
    block_size: int  # Maximum number of tokens the model can see at once.
    n_layer: int     # Number of Transformer decoder blocks.
    n_head: int      # Number of attention heads in each block.
    n_embd: int      # Width of each token vector.

    @property
    def head_dim(self):
        """Dimension handled by each attention head."""
        assert self.n_embd % self.n_head == 0, "n_embd must divide evenly by n_head"
        return self.n_embd // self.n_head


# Real-size metadata for discussion only. Do not instantiate these in NumPy.
GPT2_SMALL_REAL = GPTConfig(vocab_size=50_257, block_size=1024, n_layer=12, n_head=12, n_embd=768)
GPT3_175B_REAL = GPTConfig(vocab_size=50_257, block_size=2048, n_layer=96, n_head=96, n_embd=12_288)

# Tiny runnable versions used below.
GPT2_TOY = GPTConfig(vocab_size=64, block_size=48, n_layer=2, n_head=4, n_embd=32)
GPT3_TOY = GPTConfig(vocab_size=64, block_size=48, n_layer=4, n_head=4, n_embd=64)

print("Real GPT-2 small metadata:", GPT2_SMALL_REAL)
print("Real GPT-3 175B metadata:", GPT3_175B_REAL)
print("Toy GPT-2-like config:", GPT2_TOY)
print("Toy GPT-3-like config:", GPT3_TOY)


## 3. A Tiny Character Tokenizer

Real GPT models use subword tokenizers such as byte-pair encoding. For teaching, a character tokenizer is easier: each character becomes one integer token.

The mini corpus is built into the notebook so the demo does not require Kaggle, Colab, or internet access.


In [ ]:
DEMO_TEXT = """
medical ai can help read clinical notes.
attention lets each token look back at earlier tokens.
a gpt model predicts the next token from the previous context.
text-only models may miss information from images, waveforms, and pathology slides.
multimodal models connect language with other biomedical data.
""".strip().lower()

# Build a vocabulary from the corpus and add a fallback space character.
chars = sorted(set(DEMO_TEXT))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}


def encode(text):
    """Convert text into a NumPy array of token IDs."""
    text = text.lower()
    space_id = stoi.get(" ", 0)
    return np.array([stoi.get(ch, space_id) for ch in text], dtype=np.int64)


def decode(ids):
    """Convert token IDs back into text."""
    return "".join(itos[int(i)] for i in ids)


vocab_size = len(chars)
data_ids = encode(DEMO_TEXT)

print("Vocabulary size:", vocab_size)
print("First 20 vocabulary characters:", chars[:20])
print("Encoded data shape:", data_ids.shape)
print("Round-trip check:", decode(encode("medical ai")))


## 4. Math Utilities

These functions are small but central:

- **softmax** turns scores into probabilities.
- **GELU** is the activation function used in GPT-style feed-forward layers.
- **layer normalization** stabilizes hidden vectors at each token position.


In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def gelu(x):
    """GELU activation using the common tanh approximation."""
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))


def layer_norm(x, gamma, beta, eps=1e-5):
    """Normalize each token vector over its channel dimension."""
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_hat = (x - mean) / np.sqrt(var + eps)
    return gamma * x_hat + beta


example_scores = np.array([1.0, 2.0, 4.0])
print("Softmax example:", softmax(example_scores))


## 5. Initialize GPT Parameters

A trained GPT model is mostly a collection of learned matrices. Here we initialize them randomly so the forward pass can run.

Key matrices:

- `wte`: token embedding table
- `wpe`: positional embedding table
- `Wqkv`: one projection that creates queries, keys, and values
- `Wfc` and `Wout`: the feed-forward network
- `lm_head`: maps hidden vectors back to vocabulary logits


In [ ]:
def randn(shape, scale=0.02):
    """Small random normal initializer."""
    return (np.random.randn(*shape) * scale).astype(np.float32)


def init_block_params(config):
    """Initialize one Transformer decoder block."""
    C = config.n_embd
    return {
        "ln1_gamma": np.ones(C, dtype=np.float32),
        "ln1_beta": np.zeros(C, dtype=np.float32),
        "Wqkv": randn((C, 3 * C)),
        "bqkv": np.zeros(3 * C, dtype=np.float32),
        "Wproj": randn((C, C)),
        "bproj": np.zeros(C, dtype=np.float32),
        "ln2_gamma": np.ones(C, dtype=np.float32),
        "ln2_beta": np.zeros(C, dtype=np.float32),
        "Wfc": randn((C, 4 * C)),
        "bfc": np.zeros(4 * C, dtype=np.float32),
        "Wout": randn((4 * C, C)),
        "bout": np.zeros(C, dtype=np.float32),
    }


def init_gpt_params(config):
    """Initialize all parameters for a GPT-style model."""
    return {
        "wte": randn((config.vocab_size, config.n_embd)),
        "wpe": randn((config.block_size, config.n_embd)),
        "blocks": [init_block_params(config) for _ in range(config.n_layer)],
        "ln_f_gamma": np.ones(config.n_embd, dtype=np.float32),
        "ln_f_beta": np.zeros(config.n_embd, dtype=np.float32),
        "lm_head": randn((config.n_embd, config.vocab_size)),
    }


def count_params(obj):
    """Recursively count scalar parameters in nested dict/list structures."""
    if isinstance(obj, np.ndarray):
        return obj.size
    if isinstance(obj, dict):
        return sum(count_params(v) for v in obj.values())
    if isinstance(obj, list):
        return sum(count_params(v) for v in obj)
    return 0


## 6. Causal Multi-Head Self-Attention

Self-attention answers: **for this token, which earlier tokens are useful?**

Causal masking is what makes GPT autoregressive. Token position `t` can attend to positions `0 ... t`, but not to future positions.


In [ ]:
def make_causal_mask(T):
    """Lower-triangular mask: 1 means visible, 0 means hidden future token."""
    return np.tril(np.ones((T, T), dtype=np.float32))


print(make_causal_mask(8).astype(int))


### 6.1 Self-Attention in Plain Language

For each token, self-attention creates three learned views of the same hidden vector:

| Name | Intuition | Question it answers |
|---|---|---|
| **Query** `Q` | what this token is looking for | "What information do I need?" |
| **Key** `K` | what this token offers | "What information do I contain?" |
| **Value** `V` | the content to copy forward | "What should be passed along if I am relevant?" |

The attention score between token `i` and token `j` is roughly:

$$
\text{score}_{i,j} = \frac{Q_i \cdot K_j}{\sqrt{d_{head}}}
$$

Then softmax turns the scores into weights, and the output for token `i` is a weighted average of the value vectors from visible tokens.

In GPT, the causal mask prevents token `i` from using future tokens. That is why GPT can be trained and used for next-token prediction.


In [ ]:
# A tiny hand-readable attention example.
# Rows are tokens. Columns are toy features.
toy_tokens = ["medical", "ai", "helps"]
toy_Q = np.array([
    [1.0, 0.0],  # "medical" asks mostly about feature 1
    [0.8, 0.2],  # "ai" asks about feature 1 with a little feature 2
    [0.1, 1.0],  # "helps" asks mostly about feature 2
])
toy_K = np.array([
    [1.0, 0.0],  # "medical" offers feature 1
    [0.7, 0.3],  # "ai" offers mixed information
    [0.0, 1.0],  # "helps" offers feature 2
])
toy_V = np.array([
    [10.0, 0.0],
    [6.0, 4.0],
    [0.0, 10.0],
])

scores = toy_Q @ toy_K.T / math.sqrt(toy_Q.shape[-1])
weights_no_mask = softmax(scores, axis=-1)
output_no_mask = weights_no_mask @ toy_V

print("Tokens:", toy_tokens)
print("Raw attention scores without causal mask:")
print(scores)
print("\nAttention weights without causal mask:")
print(weights_no_mask)
print("\nWeighted value output without causal mask:")
print(output_no_mask)


### 6.2 Why the Causal Mask Matters

Without a mask, the first token could attend to later tokens. That is fine for some models, such as BERT-style encoders, but it breaks GPT-style generation because the model would peek at the future.

The mask below keeps only the lower triangle of the attention matrix.


In [ ]:
T = len(toy_tokens)
mask = make_causal_mask(T)
masked_scores = np.where(mask == 1, scores, -1e10)
weights_masked = softmax(masked_scores, axis=-1)
output_masked = weights_masked @ toy_V

print("Causal mask:")
print(mask.astype(int))
print("\nAttention weights with causal mask:")
print(weights_masked)
print("\nWeighted value output with causal mask:")
print(output_masked)


### 6.3 Masking Mechanism: What Actually Happens?

An attention mask is a simple rule applied to the attention score matrix **before softmax**.

The usual recipe is:

1. compute raw attention scores
2. replace disallowed positions with a very negative number, such as `-1e10`
3. apply softmax
4. the disallowed positions become probability `0`

This works because `exp(-1e10)` is effectively zero. The model does not delete future tokens from memory; it makes their attention probability zero.

For GPT, the most important mask is the **causal mask**:

- row `0` can see column `0`
- row `1` can see columns `0, 1`
- row `2` can see columns `0, 1, 2`
- and so on

Rows represent the token doing the looking. Columns represent the token being looked at.


In [ ]:
def print_matrix_with_labels(matrix, row_labels, col_labels, title):
    """Small helper to print attention matrices with readable labels."""
    print(title)
    print("rows = query/current token, columns = key/token being attended to")
    print(" " * 10 + " ".join(f"{c:>8s}" for c in col_labels))
    for label, row in zip(row_labels, matrix):
        print(f"{label:>8s}  " + " ".join(f"{v:8.3f}" for v in row))


mask_demo_tokens = ["the", "mass", "is", "benign"]
T = len(mask_demo_tokens)

# Use simple scores so the masking effect is easy to see.
# Before masking, every token has a score for every other token, including future tokens.
raw_scores = np.array([
    [0.8, 1.2, 0.5, 2.0],
    [0.4, 1.5, 1.0, 1.7],
    [0.2, 0.8, 1.4, 2.2],
    [0.1, 0.5, 1.0, 2.5],
], dtype=np.float32)

causal_mask = make_causal_mask(T)
masked_scores = np.where(causal_mask == 1, raw_scores, -1e10)

print_matrix_with_labels(raw_scores, mask_demo_tokens, mask_demo_tokens, "Raw attention scores before masking:")
print()
print_matrix_with_labels(causal_mask, mask_demo_tokens, mask_demo_tokens, "Causal mask, where 1 = visible and 0 = blocked:")
print()
print_matrix_with_labels(masked_scores, mask_demo_tokens, mask_demo_tokens, "Scores after applying the causal mask:")


### 6.4 Masking Before Softmax Changes the Probabilities

The mask is applied to scores, but the effect is easiest to see after softmax. Future tokens get probability zero, and the remaining visible probabilities are renormalized so each row still sums to 1.


In [ ]:
probs_without_mask = softmax(raw_scores, axis=-1)
probs_with_mask = softmax(masked_scores, axis=-1)

print_matrix_with_labels(probs_without_mask, mask_demo_tokens, mask_demo_tokens, "Attention probabilities without a causal mask:")
print("Row sums:", probs_without_mask.sum(axis=-1))
print()
print_matrix_with_labels(probs_with_mask, mask_demo_tokens, mask_demo_tokens, "Attention probabilities with a causal mask:")
print("Row sums:", probs_with_mask.sum(axis=-1))


### 6.5 Why Future Leakage Is a Problem

Suppose the training phrase is:

`the mass is benign`

If the model is predicting the next token after `the mass is`, it should not be allowed to look at `benign`. Without a causal mask, the attention mechanism could directly use the answer during training. That would make the training task artificially easy and would not match real generation, where the future token is unknown.

Causal masking makes training match generation: the model must predict the next token using only the left context.


In [ ]:
def visible_contexts(tokens):
    """Show what each token position is allowed to attend to in GPT."""
    mask = make_causal_mask(len(tokens)).astype(bool)
    for i, token in enumerate(tokens):
        visible = [tokens[j] for j in range(len(tokens)) if mask[i, j]]
        blocked = [tokens[j] for j in range(len(tokens)) if not mask[i, j]]
        print(f"Current token {i} ({token!r}) can see: {visible}")
        print(f"Blocked future tokens: {blocked}\n")


visible_contexts(mask_demo_tokens)


### 6.6 Causal Mask vs Padding Mask

There are two common kinds of masks:

| Mask type | Purpose | Example |
|---|---|---|
| **Causal mask** | blocks future tokens | GPT next-token prediction |
| **Padding mask** | blocks fake padding tokens | batching sequences of different lengths |

This notebook mainly uses causal masking. In production models, causal and padding masks are often combined so the model cannot attend to future tokens or padded positions.


In [ ]:
def make_padding_mask(lengths, max_len):
    """Return a padding mask with shape (batch, max_len). 1 = real token, 0 = padding."""
    positions = np.arange(max_len)[None, :]
    lengths = np.array(lengths)[:, None]
    return (positions < lengths).astype(np.float32)


lengths = [4, 2]
padding_mask = make_padding_mask(lengths, max_len=4)
print("Padding mask for two sequences with lengths 4 and 2:")
print(padding_mask.astype(int))
print()
print("Sequence 1 has no padding:      the mass is benign")
print("Sequence 2 has two pad tokens:  ai helps [PAD] [PAD]")


### 6.7 From One Attention Head to Multi-Head Attention

One attention head creates one pattern of token-to-token relationships. Multi-head attention repeats this process several times in parallel.

A helpful intuition:

- one head might focus on nearby words
- another head might focus on clinically important terms
- another head might track punctuation or sentence structure
- another head might connect a diagnosis phrase with a risk factor

Technically, the embedding dimension is split across heads. If `n_embd = 32` and `n_head = 4`, then each head works with `head_dim = 8` features. The heads are later concatenated and projected back to the full embedding size.


In [ ]:
def explain_attention_shapes(prompt, config):
    """Show the tensor shapes used in multi-head attention."""
    token_ids = encode(prompt)[None, :]
    B, T = token_ids.shape
    C = config.n_embd
    H = config.n_head
    D = config.head_dim

    print("Prompt:", repr(prompt))
    print("B = batch size:", B)
    print("T = sequence length:", T)
    print("C = embedding width:", C)
    print("H = number of heads:", H)
    print("D = head dimension C/H:", D)
    print()
    print("Hidden states x:              ", (B, T, C))
    print("Combined qkv projection:      ", (B, T, 3 * C))
    print("q, k, v before split_heads:   ", (B, T, C))
    print("q, k, v after split_heads:    ", (B, H, T, D))
    print("attention score matrix:       ", (B, H, T, T))
    print("attention output per head:    ", (B, H, T, D))
    print("merged attention output:      ", (B, T, C))


explain_attention_shapes("medical ai", GPT2_TOY)


### 6.8 What Multi-Head Attention Contributes to GPT

Multi-head attention gives GPT a flexible way to build contextual embeddings. The same token can mean different things depending on surrounding context, and attention is the mechanism that lets earlier tokens influence the current representation.

For biomedical language, this matters because phrases often depend on context:

- "negative" may mean good news in a cancer screen, but not in a sentiment task
- "mass" may mean a tumor, a physics quantity, or a body measurement
- abbreviations such as "CT" or "RA" require local context

Text-only attention still has a limitation: it can only attend to tokens. For radiology, pathology, waveforms, and microscopy, the model needs additional mechanisms to represent and align non-text data.


In [ ]:
def split_heads(x, n_head):
    """Reshape (B, T, C) into (B, n_head, T, head_dim)."""
    B, T, C = x.shape
    head_dim = C // n_head
    return x.reshape(B, T, n_head, head_dim).transpose(0, 2, 1, 3)


def merge_heads(x):
    """Reshape (B, n_head, T, head_dim) back into (B, T, C)."""
    B, n_head, T, head_dim = x.shape
    return x.transpose(0, 2, 1, 3).reshape(B, T, n_head * head_dim)


def causal_self_attention(x, block, config, return_weights=False):
    """Run masked multi-head self-attention.

    Args:
        x: hidden states with shape (batch, time, channels)
        block: parameters for one Transformer block
        config: model configuration
        return_weights: if True, also return attention probabilities
    """
    B, T, C = x.shape

    # One projection computes queries, keys, and values together.
    qkv = x @ block["Wqkv"] + block["bqkv"]
    q, k, v = np.split(qkv, 3, axis=-1)

    # Multiple heads let the model learn different token-token relationships.
    q = split_heads(q, config.n_head)
    k = split_heads(k, config.n_head)
    v = split_heads(v, config.n_head)

    # Attention scores compare every query token to every key token.
    scores = (q @ k.transpose(0, 1, 3, 2)) / math.sqrt(config.head_dim)

    # Hide future tokens by setting their scores to a very negative number.
    mask = make_causal_mask(T)[None, None, :, :]
    scores = np.where(mask == 1, scores, -1e10)

    # Attention probabilities sum to 1 over the key positions.
    weights = softmax(scores, axis=-1)

    # Weighted average of value vectors, then merge heads.
    attended = weights @ v
    attended = merge_heads(attended)

    # Final linear projection mixes information across heads.
    out = attended @ block["Wproj"] + block["bproj"]
    if return_weights:
        return out, weights
    return out


## 7. Transformer Decoder Block

Each GPT block has two sublayers:

1. causal self-attention
2. feed-forward MLP

Both use residual connections. GPT-2 and GPT-3 use a **pre-layer-normalization** pattern: normalize, apply the sublayer, then add the result back.


In [ ]:
def mlp(x, block):
    """Position-wise feed-forward network.

    The same two-layer MLP is applied to every token independently.
    It expands channels by 4x, applies GELU, then projects back.
    """
    h = x @ block["Wfc"] + block["bfc"]
    h = gelu(h)
    return h @ block["Wout"] + block["bout"]


def transformer_block(x, block, config):
    """One GPT decoder block."""
    # Attention sublayer.
    x_norm = layer_norm(x, block["ln1_gamma"], block["ln1_beta"])
    x = x + causal_self_attention(x_norm, block, config)

    # MLP sublayer.
    x_norm = layer_norm(x, block["ln2_gamma"], block["ln2_beta"])
    x = x + mlp(x_norm, block)
    return x


## 8. Full GPT Forward Pass

The model receives token IDs with shape `(batch, time)` and returns logits with shape `(batch, time, vocab_size)`.

A **logit** is an unnormalized score for a possible next token.


In [ ]:
def gpt_forward(token_ids, params, config, trace=False):
    """Forward pass through a GPT-style model."""
    B, T = token_ids.shape
    if T > config.block_size:
        raise ValueError(f"Sequence length {T} exceeds block_size {config.block_size}")

    # Convert token IDs to vectors and add position vectors.
    tok_emb = params["wte"][token_ids]
    pos = np.arange(T)
    pos_emb = params["wpe"][pos][None, :, :]
    x = tok_emb + pos_emb

    if trace:
        print("token_ids:", token_ids.shape)
        print("token embeddings:", tok_emb.shape)
        print("position embeddings:", pos_emb.shape)
        print("hidden state before blocks:", x.shape)

    # Repeated decoder blocks refine contextual representations.
    for i, block in enumerate(params["blocks"]):
        x = transformer_block(x, block, config)
        if trace:
            print(f"hidden state after block {i + 1}:", x.shape)

    # Final normalization and projection to vocabulary scores.
    x = layer_norm(x, params["ln_f_gamma"], params["ln_f_beta"])
    logits = x @ params["lm_head"]

    if trace:
        print("logits:", logits.shape)
    return logits


## 9. Demo: One Forward Pass

For a live demo, this is the most important cell. It shows how text becomes token IDs, how token IDs become logits, and where the vocabulary dimension appears.


In [ ]:
# Match the toy configs to this notebook's character vocabulary.
GPT2_TOY = GPTConfig(vocab_size=vocab_size, block_size=48, n_layer=2, n_head=4, n_embd=32)
GPT3_TOY = GPTConfig(vocab_size=vocab_size, block_size=48, n_layer=4, n_head=4, n_embd=64)

gpt2_params = init_gpt_params(GPT2_TOY)
gpt3_params = init_gpt_params(GPT3_TOY)

prompt = "medical ai"
x = encode(prompt)[None, :]  # add batch dimension

print("Prompt:", repr(prompt))
print("Token IDs:", x)
print()
_ = gpt_forward(x, gpt2_params, GPT2_TOY, trace=True)


### 6.9 Comparing Heads in the Random Toy Model

The model is untrained, so the heads do not yet have meaningful clinical behavior. Still, this cell is useful for teaching because it shows that each head has its own attention matrix.

After training, different heads often specialize in different relationships, although individual head interpretation should be done carefully.


In [ ]:
def show_all_head_attention(prompt, params, config):
    """Print attention matrices for all heads in the first Transformer block."""
    token_ids = encode(prompt)[None, :]
    T = token_ids.shape[1]

    tok_emb = params["wte"][token_ids]
    pos_emb = params["wpe"][np.arange(T)][None, :, :]
    x = tok_emb + pos_emb

    block = params["blocks"][0]
    x_norm = layer_norm(x, block["ln1_gamma"], block["ln1_beta"])
    _, weights = causal_self_attention(x_norm, block, config, return_weights=True)

    print("Characters:", list(prompt))
    for head in range(config.n_head):
        print(f"\nHead {head} attention weights:")
        print(weights[0, head])


show_all_head_attention("medical", gpt2_params, GPT2_TOY)


## 10. Inspect Attention Weights

Attention weights show which previous positions each token attends to. With random weights, the pattern is not meaningful yet, but the **causal structure** is visible: future positions receive zero probability.


In [ ]:
def attention_demo(prompt, params, config, head=0):
    """Print a small attention matrix for the first block and selected head."""
    token_ids = encode(prompt)[None, :]
    T = token_ids.shape[1]

    tok_emb = params["wte"][token_ids]
    pos_emb = params["wpe"][np.arange(T)][None, :, :]
    x = tok_emb + pos_emb

    block = params["blocks"][0]
    x_norm = layer_norm(x, block["ln1_gamma"], block["ln1_beta"])
    _, weights = causal_self_attention(x_norm, block, config, return_weights=True)

    print("Characters:", list(prompt))
    print("Attention matrix for batch 0, head", head)
    print(weights[0, head])


attention_demo("medical", gpt2_params, GPT2_TOY, head=0)


## 11. Predict the Next Token

The final time step contains the model's prediction for the next token after the prompt.

Because the model is untrained, these predictions are random. The point is to show the mechanics.


In [ ]:
def top_next_tokens(prompt, params, config, k=5):
    """Show the top-k next-token probabilities after a prompt."""
    token_ids = encode(prompt)[-config.block_size:]
    logits = gpt_forward(token_ids[None, :], params, config)
    next_logits = logits[0, -1]
    probs = softmax(next_logits)
    top_ids = np.argsort(probs)[-k:][::-1]

    rows = []
    for token_id in top_ids:
        rows.append((int(token_id), repr(decode([token_id])), float(probs[token_id])))
    return rows


for name, params, config in [
    ("Toy GPT-2-like", gpt2_params, GPT2_TOY),
    ("Toy GPT-3-like", gpt3_params, GPT3_TOY),
]:
    print(name)
    for token_id, token, prob in top_next_tokens("medical ai", params, config, k=5):
        print(f"  id={token_id:2d} token={token:>4s} probability={prob:.4f}")
    print()


## 12. Autoregressive Generation

Generation repeats the same loop:

1. run the model on the current context
2. take logits from the last position
3. sample one next token
4. append that token to the context

With random weights, the output is not meaningful language. After training, the same loop produces coherent text.


In [ ]:
def sample_next_token(logits, temperature=1.0, top_k=None):
    """Sample one token ID from a vector of logits."""
    logits = logits.astype(np.float64) / max(temperature, 1e-8)

    if top_k is not None:
        top_k = min(top_k, len(logits))
        cutoff = np.partition(logits, -top_k)[-top_k]
        logits = np.where(logits >= cutoff, logits, -1e10)

    probs = softmax(logits)
    return int(np.random.choice(len(probs), p=probs))


def generate(prompt, params, config, max_new_tokens=80, temperature=1.0, top_k=8):
    """Generate characters from a prompt."""
    ids = list(encode(prompt))
    if len(ids) == 0:
        raise ValueError("Prompt produced no valid tokens.")

    for _ in range(max_new_tokens):
        context = np.array(ids[-config.block_size:], dtype=np.int64)[None, :]
        logits = gpt_forward(context, params, config)
        next_logits = logits[0, -1]
        ids.append(sample_next_token(next_logits, temperature=temperature, top_k=top_k))

    return decode(ids)


print("Toy GPT-2-like sample:")
print(generate("medical ai", gpt2_params, GPT2_TOY, max_new_tokens=80, temperature=1.0, top_k=8))
print("\nToy GPT-3-like sample:")
print(generate("medical ai", gpt3_params, GPT3_TOY, max_new_tokens=80, temperature=1.0, top_k=8))


## 13. Parameter Counts Make Scaling Concrete

GPT-3-style scaling means more layers, wider embeddings, and often more heads/context. This helper counts actual NumPy parameters in our toy models.


In [ ]:
print(f"Toy GPT-2-like parameters: {count_params(gpt2_params):,}")
print(f"Toy GPT-3-like parameters: {count_params(gpt3_params):,}")
print(f"Ratio: {count_params(gpt3_params) / count_params(gpt2_params):.1f}x")


## 14. Optional: Mini-Batches for Next-Token Training

This notebook focuses on architecture and inference. If you later add backpropagation, next-token training uses pairs like this:

- input `x`: token sequence
- target `y`: the same sequence shifted one token to the left


In [ ]:
def get_batch(data_ids, batch_size=4, block_size=32):
    """Create a mini-batch for next-token prediction."""
    if len(data_ids) <= block_size + 1:
        raise ValueError("Dataset is too short for this block_size.")

    starts = np.random.randint(0, len(data_ids) - block_size - 1, size=batch_size)
    x = np.stack([data_ids[i : i + block_size] for i in starts])
    y = np.stack([data_ids[i + 1 : i + block_size + 1] for i in starts])
    return x, y


batch_x, batch_y = get_batch(data_ids, batch_size=2, block_size=32)
print("x shape:", batch_x.shape)
print("y shape:", batch_y.shape)
print("\nExample input:")
print(decode(batch_x[0]))
print("\nExample target, shifted by one token:")
print(decode(batch_y[0]))


## 15. Journal Club Discussion Prompts

1. Why does causal masking matter for text generation?
2. What does multi-head attention allow that one attention head may miss?
3. GPT-2 and GPT-3 share a similar architecture. What changed when GPT-3 scaled up?
4. Why are text-only GPT models limited in biomedical settings such as radiology or pathology?
5. What new components would be needed to extend this notebook into a multimodal model?

**Suggested live-demo path:** run Sections 1-3, pause at the causal mask, run one traced forward pass, inspect attention weights, then discuss why random architecture is not enough without training data and optimization.
